# Neuron Fate

Measure whether base-neuron formula behavior was preserved, recalibrated, or remapped after fine-tuning.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from umap import UMAP

from theoretical.activations.neuron_fate.analysis import (
    calc_bidirectional_top_k_connections,
    calc_transport_latents,
    run,
)
from theoretical.activations.source import activation_path

## Setup

In [ ]:
model_id = "LiquidAI/LFM2.5-1.2B-Thinking"
layer = "model.layers.8.feed_forward"
tasks = ("snli", "claim")
checkpoint_names = {task: None for task in tasks}
output_directory = Path("theoretical/activations/neuron_fate/data")

run_latent_views = False
run_connection_views = False
transport_rank = 64
connection_k = 5

## Complete neuron-fate measurements

In [ ]:
results_by_task = {}
artifact_arrays_by_task = {}
fate_matrices = {}

for task in tasks:
    base_path = activation_path(model_id, task, layer)
    finetuned_path = activation_path(
        model_id,
        task,
        layer,
        task_name=task,
        checkpoint_name=checkpoint_names[task],
    )
    result = run(base_path, finetuned_path)
    results_by_task[task] = result

    artifact_arrays_by_task[task] = {
        f"{group}_{name}": matrix
        for group, measurements in result.items()
        for name, matrix in measurements.items()
    }
    fate_matrices[task, "r_squared"] = result["affine"]["r_squared"]
    fate_matrices[task, "ccc"] = result["literal"]["ccc"]
    fate_matrices[task, "cross_minus_base_ccc"] = result["relative_to_base"]["cross_minus_base_ccc"]

## Save completed numerical artifacts

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
artifact_paths = {}
for task, artifact_arrays in artifact_arrays_by_task.items():
    artifact_path = output_directory / f"{task}_neuron_fate.npz"
    np.savez(artifact_path, **artifact_arrays)
    artifact_paths[task] = artifact_path

artifact_paths

## Core measurement heatmaps

In [ ]:
measurement_ranges = {
    "r_squared": ("viridis", 0.0, 1.0),
    "ccc": ("coolwarm", -1.0, 1.0),
    "cross_minus_base_ccc": ("coolwarm", -2.0, 2.0),
}

for (task, measurement), matrix in fate_matrices.items():
    cmap, vmin, vmax = measurement_ranges[measurement]
    figure, axis = plt.subplots(figsize=(10, 8))
    image = axis.imshow(
        np.ma.masked_invalid(matrix),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="equal",
    )
    axis.set_title(f"{task} {measurement}")
    axis.set_xlabel("fine-tuned neuron")
    axis.set_ylabel("base neuron")
    figure.colorbar(image, ax=axis, label=measurement)
    figure.tight_layout()
    plt.show()

## Affine recalibration

Alpha and beta are interpreted only together with the corresponding R-squared fit.

In [ ]:
for task, result in results_by_task.items():
    for measurement in ("alpha", "beta"):
        matrix = result["affine"][measurement]
        finite = matrix[np.isfinite(matrix)]
        color_limit = np.percentile(np.abs(finite), 99) if finite.size else 1.0
        color_limit = color_limit if color_limit > 0.0 else 1.0

        figure, axis = plt.subplots(figsize=(10, 8))
        image = axis.imshow(
            np.ma.masked_invalid(matrix),
            cmap="coolwarm",
            vmin=-color_limit,
            vmax=color_limit,
            interpolation="nearest",
            aspect="equal",
        )
        axis.set_title(f"{task} {measurement}")
        axis.set_xlabel("fine-tuned neuron")
        axis.set_ylabel("base neuron")
        figure.colorbar(image, ax=axis, label=measurement)
        figure.tight_layout()
        plt.show()

## Normalized bipartite spectral ordering

This is a population-structure view of a selected fate measurement, not separate evidence of transfer.

In [ ]:
ordered_fate_matrices = {}
for (task, measurement), matrix in fate_matrices.items():
    display_matrix = np.nan_to_num(matrix, nan=0.0)
    affinity = np.abs(display_matrix)
    row_degree = affinity.sum(axis=1)
    column_degree = affinity.sum(axis=0)
    normalized = np.divide(
        affinity,
        np.sqrt(row_degree[:, None] * column_degree[None, :]),
        out=np.zeros_like(affinity),
        where=(row_degree[:, None] > 0.0) & (column_degree[None, :] > 0.0),
    )
    spectral_U, spectral_s, spectral_Vt = np.linalg.svd(normalized, full_matrices=False)
    vector_index = 1 if spectral_s.size > 1 else 0
    base_order = np.argsort(spectral_U[:, vector_index], kind="stable")
    finetuned_order = np.argsort(spectral_Vt[vector_index], kind="stable")
    ordered = display_matrix[base_order][:, finetuned_order]
    ordered_fate_matrices[task, measurement] = (ordered, base_order, finetuned_order)

    cmap, vmin, vmax = measurement_ranges[measurement]
    figure, axis = plt.subplots(figsize=(10, 8))
    image = axis.imshow(ordered, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    axis.set_title(f"{task} {measurement} spectral ordering")
    axis.set_xlabel("spectrally ordered fine-tuned neuron")
    axis.set_ylabel("spectrally ordered base neuron")
    figure.colorbar(image, ax=axis, label=measurement)
    figure.tight_layout()
    plt.show()

## Singular values

In [ ]:
svd_by_task_measurement = {}
for key, matrix in fate_matrices.items():
    decomposition = np.linalg.svd(np.nan_to_num(matrix, nan=0.0), full_matrices=False)
    svd_by_task_measurement[key] = decomposition
    task, measurement = key
    singular_values = decomposition[1]

    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(np.arange(1, singular_values.size + 1), singular_values)
    axis.set_title(f"{task} {measurement} singular values")
    axis.set_xlabel("rank")
    axis.set_ylabel("singular value")
    axis.grid(True, alpha=0.3)
    figure.tight_layout()
    plt.show()

## Optional latent and connection views

These remain exploratory summaries of the selected fate matrices.

In [ ]:
if run_latent_views:
    for (task, measurement), (U, singular_values, Vt) in svd_by_task_measurement.items():
        base_latents, finetuned_latents = calc_transport_latents(
            U,
            singular_values,
            Vt,
            transport_rank,
        )
        combined = np.vstack((base_latents, finetuned_latents))
        projected = UMAP(n_components=2, random_state=0).fit_transform(combined)
        base_xy = projected[: base_latents.shape[0]]
        finetuned_xy = projected[base_latents.shape[0] :]

        figure, axis = plt.subplots(figsize=(10, 8))
        axis.scatter(base_xy[:, 0], base_xy[:, 1], s=12, label="base", alpha=0.7)
        axis.scatter(finetuned_xy[:, 0], finetuned_xy[:, 1], s=12, label="fine-tuned", alpha=0.7)
        axis.set_title(f"{task} {measurement} low-rank fate coordinates")
        axis.legend()
        figure.tight_layout()
        plt.show()

if run_connection_views:
    for (task, measurement), matrix in fate_matrices.items():
        for relationship in ("positive", "negative"):
            forward, _reverse = calc_bidirectional_top_k_connections(
                np.nan_to_num(matrix, nan=0.0),
                connection_k,
                relationship,
            )
            query_indices, candidate_indices, scores, _degrees = forward
            graph = nx.DiGraph()
            for query, candidate, score in zip(
                query_indices,
                candidate_indices,
                scores,
                strict=True,
            ):
                graph.add_edge(f"B{query}", f"F{candidate}", weight=abs(score))
            if graph.number_of_edges() == 0:
                continue
            figure, axis = plt.subplots(figsize=(12, 12))
            positions = nx.spring_layout(graph, seed=0, weight="weight")
            nx.draw_networkx(graph, positions, node_size=30, with_labels=False, ax=axis)
            axis.set_title(f"{task} {measurement} {relationship} top-{connection_k}")
            axis.axis("off")
            figure.tight_layout()
            plt.show()